# Zaino Sync Bench Analysis

Pull structured JSON logs from Loki, parse, and visualize sync engine performance.

**Setup:** Start a port-forward before running:
```bash
kubectl port-forward svc/loki 3100:3100 -n monitoring &
```

In [ ]:
import os
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from datetime import datetime, timezone
from collections import defaultdict

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

LOKI_URL = os.environ.get('LOKI_URL', 'http://127.0.0.1:3100')
print(f'Loki endpoint: {LOKI_URL}')

## 1. Fetch Logs from Loki

In [ ]:
from datetime import timedelta

def loki_query(query: str, hours_back: int = 24, page_size: int = 5000) -> list[dict]:
    """Query Loki with automatic pagination. Returns parsed JSON log lines."""
    now = datetime.now(timezone.utc)
    start_ns = int((now - timedelta(hours=hours_back)).timestamp() * 1e9)
    end_ns = int(now.timestamp() * 1e9)
    
    all_lines = []
    cursor = start_ns
    
    while cursor < end_ns:
        resp = requests.get(f'{LOKI_URL}/loki/api/v1/query_range', params={
            'query': query,
            'start': str(cursor),
            'end': str(end_ns),
            'limit': page_size,
            'direction': 'forward',
        })
        resp.raise_for_status()
        data = resp.json()
        
        page_lines = []
        max_ts = cursor
        for stream in data.get('data', {}).get('result', []):
            for ts_ns_str, line in stream.get('values', []):
                ts_ns = int(ts_ns_str)
                max_ts = max(max_ts, ts_ns)
                try:
                    obj = json.loads(line)
                    page_lines.append(obj)
                except json.JSONDecodeError:
                    pass
        
        all_lines.extend(page_lines)
        
        if not page_lines or max_ts <= cursor:
            break
        # Move cursor past the last seen timestamp
        cursor = max_ts + 1
        
        print(f'  fetched {len(all_lines)} lines so far...', end='\r')
    
    print(f'  fetched {len(all_lines)} lines total.     ')
    return all_lines


def list_pods(namespace: str = 'golden-mainnet') -> list[str]:
    """List available job pods in Loki."""
    resp = requests.get(f'{LOKI_URL}/loki/api/v1/label/pod/values')
    resp.raise_for_status()
    pods = resp.json().get('data', [])
    return [p for p in pods if 'bench' in p or 'sync' in p]


print('Available pods:', list_pods())

In [ ]:
# Configure which runs to analyze.
RUNS = {
    'old (sequential)': '{namespace="golden-mainnet",pod=~"sync-bench-tw9xp"}',
    'new (par-merge)':  '{namespace="golden-mainnet",pod=~"bench-df6f184-.*"}',
}

raw_logs = {}
for label, query in RUNS.items():
    logs = loki_query(query)
    raw_logs[label] = logs
    print(f'{label}: {len(logs)} log lines')

## 2. Parse into DataFrames

In [ ]:
def parse_duration(s: str) -> float:
    """Parse tracing duration string to milliseconds."""
    s = s.strip()
    if s.endswith('ms'):
        return float(s[:-2])
    elif s.endswith('µs') or s.endswith('us'):
        suffix_len = 2 if s.endswith('us') else len('µs')
        return float(s[:-suffix_len]) / 1000
    elif s.endswith('s'):
        return float(s[:-1]) * 1000
    return 0.0


def parse_ts(ts: str) -> datetime:
    if '.' in ts:
        base, frac = ts.split('.')
        frac = frac.rstrip('Z')[:6].ljust(6, '0')
        return datetime.fromisoformat(f'{base}.{frac}+00:00')
    return datetime.fromisoformat(ts.replace('Z', '+00:00'))


def extract_commits(logs: list[dict]) -> pd.DataFrame:
    rows = []
    for obj in logs:
        f = obj.get('fields', {})
        if f.get('message') != 'atomic batch commit':
            continue
        task_count = None
        for s in obj.get('spans', []):
            if 'task_count' in s:
                task_count = s['task_count']
                break
        rows.append({
            'timestamp': parse_ts(obj['timestamp']),
            'batch': f['batch'],
            'height': f['committed_height'],
            'op_count': f['op_count'],
            'task_count': task_count,
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values('timestamp').reset_index(drop=True)
        df['dt'] = df['timestamp'].diff().dt.total_seconds()
        df['dh'] = df['height'].diff()
        df['blocks_per_sec'] = df['dh'] / df['dt']
        df['elapsed_min'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds() / 60
    return df


def extract_merges(logs: list[dict]) -> pd.DataFrame:
    rows = []
    for obj in logs:
        f = obj.get('fields', {})
        span = obj.get('span', {})
        if span.get('name') != 'merge_persist' or f.get('message') != 'close':
            continue
        rows.append({
            'timestamp': parse_ts(obj['timestamp']),
            'index': span.get('index', '?'),
            'batch': span.get('batch', -1),
            'duration_ms': parse_duration(f.get('time.busy', '0us')),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values('timestamp').reset_index(drop=True)
    return df


def extract_provisioner(logs: list[dict]) -> pd.DataFrame:
    rows = []
    for obj in logs:
        f = obj.get('fields', {})
        if f.get('message') != 'provisioner progress':
            continue
        rows.append({
            'timestamp': parse_ts(obj['timestamp']),
            'sent': f['sent'],
            'total': f['total'],
            'blocks_per_sec': float(f['blocks_per_sec']),
        })
    return pd.DataFrame(rows)


# Parse all runs
commits = {}
merges = {}
provisioner = {}

for label, logs in raw_logs.items():
    commits[label] = extract_commits(logs)
    merges[label] = extract_merges(logs)
    provisioner[label] = extract_provisioner(logs)
    c = commits[label]
    if not c.empty:
        total_time = (c['timestamp'].iloc[-1] - c['timestamp'].iloc[0]).total_seconds()
        total_blocks = c['height'].iloc[-1] - c['height'].iloc[0]
        rate = total_blocks / total_time if total_time > 0 else 0
        print(f'{label}: {c["height"].iloc[0]:,} -> {c["height"].iloc[-1]:,}  '
              f'({total_time/60:.1f}min, {rate:.0f} blk/s overall)')

## 3. Throughput Over Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: throughput vs wall time
ax = axes[0]
for label, df in commits.items():
    if df.empty:
        continue
    valid = df.dropna(subset=['blocks_per_sec'])
    ax.plot(valid['elapsed_min'], valid['blocks_per_sec'].rolling(10).mean(),
            label=label, alpha=0.8)
ax.set_xlabel('Wall Time (min)')
ax.set_ylabel('blocks/s (rolling avg 10)')
ax.set_title('Throughput vs Wall Time')
ax.legend()
ax.grid(alpha=0.3)

# Right: throughput vs chain height
ax = axes[1]
for label, df in commits.items():
    if df.empty:
        continue
    valid = df.dropna(subset=['blocks_per_sec'])
    ax.plot(valid['height'], valid['blocks_per_sec'].rolling(10).mean(),
            label=label, alpha=0.8)
ax.set_xlabel('Chain Height')
ax.set_ylabel('blocks/s (rolling avg 10)')
ax.set_title('Throughput vs Chain Height')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Throughput by Height Band

In [ ]:
BAND_SIZE = 500_000

fig, ax = plt.subplots(figsize=(14, 5))
width = 0.35

all_bands = set()
band_data = {}
for label, df in commits.items():
    if df.empty:
        continue
    valid = df.dropna(subset=['blocks_per_sec']).copy()
    valid['band'] = (valid['height'] // BAND_SIZE) * BAND_SIZE
    grouped = valid.groupby('band')['blocks_per_sec'].agg(['mean', 'median', 'std', 'count'])
    band_data[label] = grouped
    all_bands |= set(grouped.index)

all_bands = sorted(all_bands)
x = np.arange(len(all_bands))
labels_list = list(band_data.keys())

for i, (label, grouped) in enumerate(band_data.items()):
    offset = (i - len(labels_list) / 2 + 0.5) * width
    means = [grouped.loc[b, 'mean'] if b in grouped.index else 0 for b in all_bands]
    stds = [grouped.loc[b, 'std'] if b in grouped.index else 0 for b in all_bands]
    ax.bar(x + offset, means, width, yerr=stds, label=label, alpha=0.8, capsize=3)

ax.set_xlabel('Chain Height Band')
ax.set_ylabel('blocks/s')
ax.set_title('Mean Throughput by Height Band')
ax.set_xticks(x)
ax.set_xticklabels([f'{b//1000}k' for b in all_bands], rotation=45)
ax.legend()
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Batch Complexity (op_count)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ops per batch vs height
ax = axes[0]
for label, df in commits.items():
    if df.empty:
        continue
    ax.scatter(df['height'], df['op_count'], s=2, alpha=0.4, label=label)
ax.set_xlabel('Chain Height')
ax.set_ylabel('op_count per batch')
ax.set_title('Batch Complexity vs Height')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.legend()
ax.grid(alpha=0.3)

# throughput vs op_count scatter
ax = axes[1]
for label, df in commits.items():
    if df.empty:
        continue
    valid = df.dropna(subset=['blocks_per_sec'])
    ax.scatter(valid['op_count'], valid['blocks_per_sec'], s=2, alpha=0.4, label=label)
ax.set_xlabel('op_count per batch')
ax.set_ylabel('blocks/s')
ax.set_title('Throughput vs Batch Complexity')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Parallelism (task_count)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

for label, df in commits.items():
    if df.empty or 'task_count' not in df.columns:
        continue
    valid = df.dropna(subset=['task_count'])
    ax.plot(valid['height'], valid['task_count'], '.', markersize=2, alpha=0.5, label=label)

ax.set_xlabel('Chain Height')
ax.set_ylabel('task_count')
ax.set_title('Parallelism Over Chain Height')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Per-Index Merge Duration

In [ ]:
for label, df in merges.items():
    if df.empty:
        print(f'{label}: no merge events')
        continue

    indexes = sorted(df['index'].unique())
    fig, axes = plt.subplots(1, 1, figsize=(14, 5))

    data_for_box = [df[df['index'] == idx]['duration_ms'].values for idx in indexes]
    bp = axes.boxplot(data_for_box, tick_labels=indexes, vert=True, patch_artist=True,
                      showfliers=False)
    axes.set_ylabel('duration (ms)')
    axes.set_title(f'Merge+Persist Duration by Index \u2014 {label}')
    axes.tick_params(axis='x', rotation=30)
    axes.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

In [ ]:
# Merge duration over time for each run (line per index)
for label, df in merges.items():
    if df.empty:
        continue

    fig, ax = plt.subplots(figsize=(14, 5))
    for idx in sorted(df['index'].unique()):
        sub = df[df['index'] == idx].copy()
        # Use batch as x-axis for alignment
        rolled = sub.set_index('batch')['duration_ms'].rolling(20).mean()
        ax.plot(rolled.index, rolled.values, label=idx, alpha=0.8)

    ax.set_xlabel('Batch')
    ax.set_ylabel('duration (ms, rolling avg 20)')
    ax.set_title(f'Merge Duration Over Time — {label}')
    ax.legend(loc='upper left', fontsize='small')
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

## 8. Summary Table

In [ ]:
summary_rows = []
for label, df in commits.items():
    if df.empty:
        continue
    valid = df.dropna(subset=['blocks_per_sec'])
    total_time = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds()
    total_blocks = df['height'].iloc[-1] - df['height'].iloc[0]
    
    mdf = merges.get(label, pd.DataFrame())
    merge_p50 = mdf['duration_ms'].median() if not mdf.empty else None
    merge_p95 = mdf['duration_ms'].quantile(0.95) if not mdf.empty else None
    
    summary_rows.append({
        'run': label,
        'height_from': df['height'].iloc[0],
        'height_to': df['height'].iloc[-1],
        'batches': len(df),
        'wall_min': total_time / 60,
        'overall_bps': total_blocks / total_time if total_time > 0 else 0,
        'p50_bps': valid['blocks_per_sec'].median(),
        'p95_bps': valid['blocks_per_sec'].quantile(0.95),
        'merge_p50_ms': merge_p50,
        'merge_p95_ms': merge_p95,
    })

summary = pd.DataFrame(summary_rows)
summary.style.format({
    'height_from': '{:,.0f}',
    'height_to': '{:,.0f}',
    'wall_min': '{:.1f}',
    'overall_bps': '{:.0f}',
    'p50_bps': '{:.0f}',
    'p95_bps': '{:.0f}',
    'merge_p50_ms': '{:.1f}',
    'merge_p95_ms': '{:.1f}',
})